In [1]:
import numpy as np
import pandas as pd
import re
import string
import pickle

In [2]:
def remove_punctuations(text):
    for punctuation in string.punctuation:
        text = text.replace(punctuation, '')
    return text

In [3]:
with open('../static/model/model.pickle', 'rb') as f:
    model = pickle.load(f)

In [4]:
with open('../static/model/corpora/stopwords/english', 'r') as file:
    sw = file.read().splitlines()

In [5]:
vocab = pd.read_csv('../static/model/vocabulary.txt', header=None)
tokens = vocab[0].tolist()

In [6]:
from nltk.stem import PorterStemmer
ps = PorterStemmer()

In [7]:
def preprocessing(text):
    data = pd.DataFrame([text], columns=['review'])
    data["review"] = data["review"].apply(lambda x: " ".join(x.lower() for x in x.split()))
    data["review"] = data['review'].apply(lambda x: " ".join(re.sub(r'^https?:\/\/.*[\r\n]*', '', x, flags=re.MULTILINE) for x in x.split()))
    data["review"] = data["review"].apply(remove_punctuations)
    data["review"] = data['review'].str.replace('\d+', '', regex=True)
    data["review"] = data["review"].apply(lambda x: " ".join(x for x in x.split() if x not in sw))
    data["review"] = data["review"].apply(lambda x: " ".join(ps.stem(x) for x in x.split()))
    return data["review"]

<>:6: SyntaxWarning: invalid escape sequence '\d'
<>:6: SyntaxWarning: invalid escape sequence '\d'
C:\Users\Chirath Rasandun\AppData\Local\Temp\ipykernel_16496\3591628485.py:6: SyntaxWarning: invalid escape sequence '\d'
  data["review"] = data['review'].str.replace('\d+', '', regex=True)


In [23]:
with open('../static/model/tfidf_vectorizer.pickle', 'rb') as f:
    vectorizer = pickle.load(f)

In [24]:
def vectorize_text(ds):
    return vectorizer.transform(ds)

In [35]:
def predict(text):
    processed_text = preprocessing(text)
    vectorized_text = vectorize_text(processed_text)
    prediction = model.predict(vectorized_text)[0]
    probability = model.predict_proba(vectorized_text)[0]
    
    # Get probability of predicted class
    class_index = list(model.classes_).index(prediction)
    confidence = probability[class_index]
    
    output_str = f'"text": "{text}",\n"sentiment": "{prediction}",\n"confidence": {round(float(confidence), 2)}'
    return output_str

In [37]:
txt = "I absolutely love how super and good story it have"

result = predict(txt)
print(result)

"text": "I absolutely love how super and good story it have",
"sentiment": "positive",
"confidence": 0.9
